In [125]:
import numpy as np
import pandas as pd
import os, re

In [6]:
mmlu_pro_M1 = pd.read_csv("data/mmlu-pro/M1.csv")
mmlu_pro_M2 = pd.read_csv("data/mmlu-pro/M2.csv")
agg_M1 = pd.read_csv("data/bbh+gpqa+ifeval+math+musr/M1.csv")
agg_M2 = pd.read_csv("data/bbh+gpqa+ifeval+math+musr/M2.csv")

In [72]:
# GPQA (all): N_q = 1192, MMLU-Pro: N_q = 12032
gpqa_all_M1 = agg_M1[["model", "created_date", "sha"] + [c for c in agg_M1.columns if "gpqa" in c]]
gpqa_all_M2 = agg_M2[["model", "created_date", "sha"] + [c for c in agg_M2.columns if "gpqa" in c]]
mmlu_pro_renamed_M1 = mmlu_pro_M1.rename(columns={str(i) : f"mmlu_pro_{i}" for i in range(12032)})
mmlu_pro_renamed_M2 = mmlu_pro_M2.rename(columns={str(i) : f"mmlu_pro_{i}" for i in range(12032)})

# concatenate together + store to a directory (the 100% observed versions)
os.makedirs("data/mmlu-pro-and-gpqa-all", exist_ok=True)
concat_M1 = pd.merge(mmlu_pro_renamed_M1, gpqa_all_M1)
concat_M2 = pd.merge(mmlu_pro_renamed_M2, gpqa_all_M2)
concat_M1.to_csv("data/mmlu-pro-and-gpqa-all/M1.csv", index=False)
concat_M2.to_csv("data/mmlu-pro-and-gpqa-all/M2.csv", index=False)

In [120]:
# set a seed for reproducibility
np.random.seed(858)

# apply a mask to only observe 0.1% of concat_M1's GPQA entries (i.e., simulating unobserved task)
n_models, n_questions = len(concat_M1.index), len(gpqa_all_M1.columns) - 3 # 3 metadata columns.
obs_mask = np.random.binomial(n=1, p=0.001, size=(n_models, n_questions)).astype(bool)

# nan out 99.9% of the historical GPQA outcomes (only baselines get access to it!)
gpqa_outcomes_M1 = concat_M1[[c for c in concat_M1.columns if "gpqa" in c]].to_numpy()
gpqa_outcomes_M1[~obs_mask] = np.nan

# make a copy of concat_M1 + apply the masking
concat_M1_masked = concat_M1.copy(deep=True)
concat_M1_masked.iloc[:,-n_questions:] = gpqa_outcomes_M1

# save the masked version too
concat_M1_masked.to_csv("data/mmlu-pro-and-gpqa-all/M1_p=0.001.csv", index=False)